# M2-01: Document Structure Generation Experiment

Экспериментальный notebook для разработки и тестирования генерации структуры документа.

**Цель**: Разработать промпт и модель для генерации структуры всего документа за один вызов LLM.

**Входные данные**:
- Тема/вопрос пользователя
- Опционально: конспекты пользователя

**Выходные данные**:
- DocumentStructure с несколькими секциями и подразделами

**Принцип**: Все секции генерируются одновременно для обеспечения консистентности между ними.

## 1. Setup

In [1]:
import os
import sys
from pathlib import Path
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# Загружаем переменные окружения
project_root = Path().cwd().parent.parent
env_local = project_root / ".env.local"
env_file = project_root / ".env"

if env_local.exists():
    load_dotenv(env_local)
    print(f"✓ Loaded .env.local from {env_local}")
elif env_file.exists():
    load_dotenv(env_file)
    print(f"✓ Loaded .env from {env_file}")
else:
    print("⚠ No .env file found")

# Проверяем API ключ
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment")

print(f"✓ API key loaded: {api_key[:8]}...")

MODEL_NAME = "gpt-4.1-mini"

✓ Loaded .env.local from /home/bbaron/dev/my_pet_projects/learnflow-ai/.env.local
✓ API key loaded: sk-proj-...


In [2]:
def pretty_print_pydantic(model):
    print(model.model_dump_json(indent=4))

## 2. Pydantic Models (Structured Output)

In [3]:
class Subsection(BaseModel):
    """Document subsection"""
    title: str = Field(description="Subsection title")
    theses: List[str] = Field(
        default_factory=list,
        description="Key theses to be covered in this subsection"
    )


class Section(BaseModel):
    """Document section"""
    title: str = Field(description="Section title")
    subsections: List[Subsection] = Field(
        default_factory=list,
        description="List of subsections"
    )


class DocumentStructure(BaseModel):
    """Document structure"""
    sections: List[Section] = Field(description="List of document sections")


# Тестовый пример структуры
test_structure = DocumentStructure(
    sections=[
        Section(
            title="Основные понятия квантовой механики",
            subsections=[
                Subsection(
                    title="Волновая функция",
                    theses=[
                        "Математическое описание состояния квантовой системы",
                        "Физический смысл и интерпретация волновой функции"
                    ]
                ),
                Subsection(title="Принцип неопределенности Гейзенберга", theses=[]),
                Subsection(title="Квантовые числа", theses=[])
            ]
        ),
        Section(
            title="Уравнение Шрёдингера",
            subsections=[
                Subsection(title="Временное уравнение Шрёдингера", theses=[]),
                Subsection(title="Стационарное уравнение Шрёдингера", theses=[])
            ]
        )
    ]
)

print("✓ Models defined successfully")
print(f"\nTest structure has {len(test_structure.sections)} sections:")
for i, section in enumerate(test_structure.sections, 1):
    print(f"  {i}. {section.title} ({len(section.subsections)} subsections)")

✓ Models defined successfully

Test structure has 2 sections:
  1. Основные понятия квантовой механики (3 subsections)
  2. Уравнение Шрёдингера (2 subsections)


In [4]:
print(pretty_print_pydantic(test_structure))

{
    "sections": [
        {
            "title": "Основные понятия квантовой механики",
            "subsections": [
                {
                    "title": "Волновая функция",
                    "theses": [
                        "Математическое описание состояния квантовой системы",
                        "Физический смысл и интерпретация волновой функции"
                    ]
                },
                {
                    "title": "Принцип неопределенности Гейзенберга",
                    "theses": []
                },
                {
                    "title": "Квантовые числа",
                    "theses": []
                }
            ]
        },
        {
            "title": "Уравнение Шрёдингера",
            "subsections": [
                {
                    "title": "Временное уравнение Шрёдингера",
                    "theses": []
                },
                {
                    "title": "Стационарное уравнение Шрёдингера",
    

## 3. Placeholders Analysis for System Prompt

### Используемые плейсхолдеры (WILL USE):

**Базовая персонализация:**
- `{{ subject_keywords }}`
- `{{ role_perspective }}`
- `{{ subject_name }}`
- `{{ language }}`

**Целевая аудитория:**
- `{{ target_audience_inline }}`
- `{{ target_audience_block }}`

**Охват и тип материала:**
- `{{ topic_coverage }}`
- `{{ material_type_inline }}`

**Входные данные из workflow state:**
- `{{ input_content }}`
- `{{ recognized_notes }}`

**Content generation specific:**
- `{{ explanation_depth }}`
- `{{ style }}`
- `{{ material_type_block }}`

---

### НЕ используемые плейсхолдеры (WON'T USE):

**Question-related:**
- `{{ question_formats }}`
- `{{ question_purpose }}`
- `{{ question_purpose_inline }}`
- `{{ question_quantity }}`

## 4. Placeholder Configuration

Load placeholder values from `initial_data.yaml` to avoid duplication.

In [6]:
import yaml

# Load placeholder configurations from initial_data.yaml
config_path = project_root / "prompt-config-service" / "initial_data.yaml"

with open(config_path, 'r', encoding='utf-8') as f:
    config_data = yaml.safe_load(f)

# Helper function to get placeholder value by name
def get_placeholder_value(placeholder_name: str, value_name: str) -> str:
    """
    Get placeholder value from config by placeholder name and value name.
    
    Args:
        placeholder_name: Name of the placeholder (e.g., 'subject_name')
        value_name: Name of the specific value (e.g., 'machine_learning')
    
    Returns:
        Value string from config
    """
    for placeholder in config_data['placeholders']:
        if placeholder['name'] == placeholder_name:
            for value in placeholder['values']:
                if value['name'] == value_name:
                    return value['value']
    raise ValueError(f"Value '{value_name}' not found for placeholder '{placeholder_name}'")

# Helper function to build placeholder dict from profile and custom values
def build_placeholders(
    subject_profile: str,
    style_profile: str, 
    language_value: str = "russian_tech",
    input_content: str = "",
    external_sources: str = ""
) -> dict:
    """
    Build placeholder dictionary from profile names.
    
    Args:
        subject_profile: Subject profile name (e.g., 'machine_learning_profile')
        style_profile: Style profile name (e.g., 'intermediate_level')
        language_value: Language value name (default: 'russian_tech')
        input_content: User input/topic
        external_sources: External sources content
    
    Returns:
        Dictionary with all placeholder values
    """
    placeholders = {}
    
    # Get profile settings
    subject_settings = None
    style_settings = None
    
    for profile in config_data['profiles']:
        if profile['name'] == subject_profile:
            subject_settings = profile['settings']
        elif profile['name'] == style_profile:
            style_settings = profile['settings']
    
    if not subject_settings:
        raise ValueError(f"Subject profile '{subject_profile}' not found")
    if not style_settings:
        raise ValueError(f"Style profile '{style_profile}' not found")
    
    # Merge settings (style overrides subject for overlapping keys)
    merged_settings = {**subject_settings, **style_settings}
    
    # Resolve all placeholder values
    for key, value_name in merged_settings.items():
        placeholders[key] = get_placeholder_value(key, value_name)
    
    # Add language
    placeholders['language'] = get_placeholder_value('language', language_value)
    
    # Add input data
    placeholders['input_content'] = input_content
    placeholders['external_sources'] = external_sources
    
    return placeholders

print("✓ Placeholder loading utilities defined")

✓ Placeholder loading utilities defined


In [13]:
# Example: Build placeholders for testing
# Subject: Machine Learning, Style: Intermediate Level

test_placeholders = build_placeholders(
    subject_profile="machine_learning_profile",
    style_profile="intermediate_level",
    language_value="russian_tech",
    input_content="Векторная база Qdrant: полный фундамент для production-использования",
    external_sources=""
)

# Display the configuration
print("Test Configuration:")
print("=" * 60)
print(f"Subject: {test_placeholders['subject_name']}")
print(f"Role: {test_placeholders['role_perspective']}")
print(f"Audience: {test_placeholders['target_audience_inline']}")
print(f"Depth: {test_placeholders['explanation_depth']}")
print(f"Coverage: {test_placeholders['topic_coverage']}")
print(f"Language: {test_placeholders['language']}")
print("=" * 60)
print("\nAll placeholder keys:")
for key in sorted(test_placeholders.keys()):
    print(f"  - {key}")

Test Configuration:
Subject: machine learning
Role: industry expert with deep practical understanding and applied knowledge
Audience: specialists with foundational understanding seeking deeper conceptual knowledge
Depth: intermediate depth explaining mechanisms and causal relationships
Coverage: focused on essential principles and key mechanisms
Language: russian with preserved english technical terms and abbreviations

All placeholder keys:
  - explanation_depth
  - external_sources
  - input_content
  - language
  - material_type_block
  - material_type_inline
  - question_formats
  - question_purpose
  - question_purpose_inline
  - question_quantity
  - role_perspective
  - style
  - subject_keywords
  - subject_name
  - target_audience_block
  - target_audience_inline
  - topic_coverage


In [ ]:
## 4. System Prompt for Document Structure Planning

PLANNING_STRUCTURE_SYSTEM_PROMPT = """
KEYWORD: {{ subject_keywords }}
<!-- Keywords above activate domain expertise, use naturally if relevant -->

<role>
You are a {{ role_perspective }} specializing in {{ subject_name }}, designing document structures for {{ target_audience_inline }}.
</role>

<task>
Design a hierarchical document structure (sections with subsections and theses) that serves as a blueprint for comprehensive {{ material_type_inline }} generation.
</task>

<input_data>
  <topic>
  {{ input_content }}
  </topic>

  <external_sources>
  {{ external_sources }}
  </external_sources>
</input_data>

<structure_requirements>
  <source_integration>
    - Every concept, topic, method, or detail from <external_sources></external_sources> MUST be reflected in the structure
    - Map each element from sources to appropriate sections, subsections or theses
    - Complement sources with your domain knowledge to fill gaps and provide context
    - Ensure coherent synthesis between all available information
    - If no external sources provided, rely entirely on your domain expertise
  </source_integration>

  <hierarchy_design>
    - Design structure depth and breadth naturally fitting the topic scope and context
    - Number of sections should emerge organically (may be 1-2 for focused topics, or more for comprehensive coverage)
    - Adapt structure complexity to align with explanation depth, coverage scope, and audience needs specified below
    - Use clear, specific, self-descriptive titles
    - Maintain consistent naming style throughout
    - Avoid overly generic or vague titles
    - Each subsection represents a focused learning unit covering a specific aspect
    - Each section groups related subsections into a coherent thematic block
  </hierarchy_design>

  <content_parameters>
    <topic_coverage> {{ topic_coverage }} </topic_coverage>
    <explanation_depth> {{ explanation_depth }} </explanation_depth>
    <style> {{ style }} </style>
    <material_type> {{ material_type_block }} </material_type>
  </content_parameters>

  <target_audience> {{ target_audience_block }} </target_audience>
</structure_requirements>

<output_format>
  <language> {{ language }} </language>
</output_format>
"""

print("✓ System prompt defined")

## 4. Generation Function

In [11]:
from jinja2 import Template

def generate_document_structure(
    placeholders: dict,
    model_name: str = "gpt-4o-mini",
    temperature: float = 0.3,
    verbose: bool = True
) -> DocumentStructure:
    """
    Генерирует структуру всего документа за один вызов LLM.
    
    Args:
        placeholders: Dictionary with all placeholder values
        model_name: Название модели OpenAI
        temperature: Температура генерации
        verbose: Выводить ли промежуточную информацию
    
    Returns:
        DocumentStructure со всеми секциями
    """
    # Создаем модель с structured output
    llm = ChatOpenAI(
        model=model_name,
        temperature=temperature,
        api_key=api_key
    )
    
    structured_llm = llm.with_structured_output(DocumentStructure)
    
    # Рендерим системный промпт через Jinja2
    template = Template(PLANNING_STRUCTURE_SYSTEM_PROMPT)
    rendered_prompt = template.render(**placeholders)
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"Generating document structure")
        print(f"Topic: {placeholders['input_content'][:80]}{'...' if len(placeholders['input_content']) > 80 else ''}")
        print(f"Model: {model_name} (temp={temperature})")
        print(f"External sources: {'Yes' if placeholders['external_sources'] else 'No'}")
        print(f"{'='*60}\n")
    
    # Генерируем
    result = structured_llm.invoke(rendered_prompt)
    
    if verbose:
        print(f"✓ Generated structure with {len(result.sections)} sections")
        total_subsections = sum(len(s.subsections) for s in result.sections)
        print(f"✓ Total subsections: {total_subsections}")
    
    return result


print("✓ Generation function defined")

✓ Generation function defined


In [14]:
# Test: Generate structure for ML/AI topic with intermediate depth
structure = generate_document_structure(
    placeholders=test_placeholders,
    model_name=MODEL_NAME,
    temperature=0.3
)

# Display the generated structure
print("\n" + "="*80)
print("GENERATED DOCUMENT STRUCTURE")
print("="*80 + "\n")

for i, section in enumerate(structure.sections, 1):
    print(f"\n{i}. {section.title}")
    print("-" * 70)
    for j, subsection in enumerate(section.subsections, 1):
        print(f"\n  {i}.{j}. {subsection.title}")
        if subsection.theses:
            for thesis in subsection.theses:
                print(f"      • {thesis}")
        else:
            print(f"      (no theses)")

print("\n" + "="*80)


Generating document structure
Topic: Векторная база Qdrant: полный фундамент для production-использования
Model: gpt-4.1-mini (temp=0.3)
External sources: No

✓ Generated structure with 4 sections
✓ Total subsections: 11

GENERATED DOCUMENT STRUCTURE


1. Введение в векторные базы данных и Qdrant
----------------------------------------------------------------------

  1.1. Основы векторных представлений данных
      • Понятие векторных представлений и их роль в машинном обучении
      • Связь между векторными представлениями и семантическим поиском
      • Почему векторные базы данных необходимы для современных AI-систем

  1.2. Обзор Qdrant как векторной базы данных
      • Архитектура и ключевые компоненты Qdrant
      • Позиционирование Qdrant на рынке векторных баз данных
      • Основные возможности и преимущества Qdrant для production-использования

2. Архитектура и ключевые механизмы Qdrant
----------------------------------------------------------------------

  2.1. Хранение

## 6. Test with External Sources

## 5. Test Generation

## 5. Example 1: Structure generation without notes

In [ ]:
# Определяем тему
topic_1 = "Векторная база Qdrant: полный фундамент для production-использования"

# Генерируем структуру документа
structure_1 = generate_document_structure(
    topic=topic_1,
    recognized_notes="",
    model_name=MODEL_NAME,
    temperature=0.3
)

# Выводим результат
print("\n" + "="*60)
print("GENERATED DOCUMENT STRUCTURE:")
print("="*60 + "\n")

for i, section in enumerate(structure_1.sections, 1):
    print(f"{i}. {section.title}")
    for j, subsection in enumerate(section.subsections, 1):
        if subsection.description:
            print(f"   {i}.{j}. {subsection.title}")
            print(f"        → {subsection.description}")
        else:
            print(f"   {i}.{j}. {subsection.title}")
    print()

## 6. Example 2: Structure generation with handwritten notes

In [ ]:
# Симулируем распознанные конспекты
sample_notes = """
### 1. Qdrant — распределённая и гибкая векторная база данных

- Тип: Open Source
- Алгоритмы поиска: HNSW (собственная реализация)
- Фильтрация по метаданным: Гибкая, поддерживает вложенную структуру и сложные иерархии
- Масштабирование: Да, поддерживается шардинг и репликация
- Квантование: Да, продуктовое и скалярное
- Use-case: Production-ready решения, работа с большими объёмами данных
- Сложность развёртывания: Средняя (требуется настройка кластеров и шардинга)

Описание:  
Qdrant написан на Rust, подходит для продакшн-сценариев и больших объёмов данных. Поддерживает полноценную фильтрацию по метаданным, включая вложенные структуры и сложные фильтры.

---

### 2. Пример 1: NimbleNote — поиск заметок

- Исходные данные:
  - 1-10 млн эмбеддингов
  - Низкий QPS (<30)
  - Простой стек, 1 сервер
  - Важна быстрая MVP, минимальный DevOps

- Выбор: Chroma  
  - Лёгкая установка, не требует сложной настройки

- Новые условия:
  - Гибкая фильтрация по метаданным (сложные AND/OR, range-фильтры)
  - Необходимость горизонтального масштабирования

- Переход на: Qdrant
"""

# Генерируем структуру с учетом конспектов
structure_2 = generate_document_structure(
    topic=topic_1,
    recognized_notes=sample_notes,
    model_name=MODEL_NAME,
    temperature=0.3
)

# Выводим результат
print("\n" + "="*60)
print("GENERATED DOCUMENT STRUCTURE (with notes):")
print("="*60 + "\n")

for i, section in enumerate(structure_2.sections, 1):
    print(f"{i}. {section.title}")
    for j, subsection in enumerate(section.subsections, 1):
        if subsection.description:
            print(f"   {i}.{j}. {subsection.title}")
            print(f"        → {subsection.description}")
        else:
            print(f"   {i}.{j}. {subsection.title}")
    print()

## 7. Comparison: Structure with vs without notes

In [ ]:
print("COMPARISON:")
print(f"\nWithout notes:")
print(f"  - Sections: {len(structure_1.sections)}")
print(f"  - Total subsections: {sum(len(s.subsections) for s in structure_1.sections)}")

print(f"\nWith notes:")
print(f"  - Sections: {len(structure_2.sections)}")
print(f"  - Total subsections: {sum(len(s.subsections) for s in structure_2.sections)}")

# Сравнение покрытия тем из конспектов
print("\n" + "="*60)
print("Coverage analysis (topics from notes):")
print("="*60)

note_topics = ["Волновая функция", "Шрёдингер", "Операторы", "Коммутатор", "неопределенности"]

for topic_keyword in note_topics:
    # Проверяем наличие в структуре с конспектами
    found_in_structure2 = False
    for section in structure_2.sections:
        if topic_keyword.lower() in section.title.lower():
            found_in_structure2 = True
            break
        for subsection in section.subsections:
            if topic_keyword.lower() in subsection.title.lower():
                found_in_structure2 = True
                break
    
    print(f"  '{topic_keyword}': {'✓ covered' if found_in_structure2 else '✗ not found'}")

In [ ]:
from typing import Literal

class DocumentStructureHITL(BaseModel):
    """
    Enhanced DocumentStructure для HITL итераций с control flow.
    """
    next_step: Literal["clarify", "finalize"] = Field(
        description=(
            "Control flow decision: "
            "'clarify' - user wants refinements, continue HITL loop; "
            "'finalize' - user approves structure, exit HITL and proceed to generation"
        )
    )
    sections: List[Section] = Field(
        description="List of document sections (refined based on user feedback)"
    )


# Тестовый пример HITL схемы
test_hitl_structure = DocumentStructureHITL(
    next_step="clarify",
    sections=[
        Section(
            title="Основы Qdrant",
            subsections=[
                Subsection(title="Архитектура Qdrant", theses=[]),
                Subsection(title="Индексирование (HNSW)", theses=[])
            ]
        )
    ]
)

print("✓ HITL schemas defined")
print(f"\nTest HITL structure:")
print(f"  next_step: {test_hitl_structure.next_step}")
print(f"  sections: {len(test_hitl_structure.sections)}")

### 8.1. Structured Output для HITL

## 8. HITL Pattern: Human-in-the-Loop для итеративной доработки структуры

**Цель:** Реализовать паттерн feedback для улучшения структуры документа на основе пользовательских комментариев.

**Подход:** Dual-prompt pattern (initial + further) по аналогии с `generating_questions`.

In [ ]:
PLANNING_STRUCTURE_FURTHER_SYSTEM_PROMPT = """
KEYWORD: {{ subject_keywords }}
<!-- Keywords above activate domain expertise, use naturally if relevant -->

<role>
You are a {{ role_perspective }} specializing in {{ subject_name }}, designing document structures for {{ target_audience_inline }}.
</role>

<task>
Refine the existing document structure based on user feedback, improving organization, coverage, or focus to better meet their needs.
</task>

<input_data>
  <topic>
  {{ input_content }}
  </topic>

  <external_sources>
  {{ external_sources }}
  </external_sources>
  
  <current_structure>
  {{ current_structure }}
  </current_structure>
</input_data>

<structure_requirements>
  <source_integration>
    - Every concept, topic, method, or detail from <external_sources></external_sources> MUST be reflected in the structure
    - Map each element from sources to appropriate sections, subsections or theses
    - Complement sources with your domain knowledge to fill gaps and provide context
    - Ensure coherent synthesis between all available information
    - If no external sources provided, rely entirely on your domain expertise
  </source_integration>

  <hierarchy_design>
    - Design structure depth and breadth naturally fitting the topic scope and context
    - Number of sections should emerge organically (may be 1-2 for focused topics, or more for comprehensive coverage)
    - Adapt structure complexity to align with explanation depth, coverage scope, and audience needs specified below
    - Use clear, specific, self-descriptive titles
    - Maintain consistent naming style throughout
    - Avoid overly generic or vague titles
    - Each subsection represents a focused learning unit covering a specific aspect
    - Each section groups related subsections into a coherent thematic block
  </hierarchy_design>

  <content_parameters>
    <topic_coverage> {{ topic_coverage }} </topic_coverage>
    <explanation_depth> {{ explanation_depth }} </explanation_depth>
    <style> {{ style }} </style>
    <material_type> {{ material_type_block }} </material_type>
  </content_parameters>

  <target_audience> {{ target_audience_block }} </target_audience>
</structure_requirements>

<refinement_requirements>
  <user_interaction>
    - Carefully analyze conversation history to understand requested changes
    - Apply user feedback precisely to improve structure organization
    - Maintain relevance to the original topic and external sources
    - Preserve successful aspects of current structure while addressing weak points
  </user_interaction>

  <approval_detection>
    - If user expresses satisfaction (e.g., "всё хорошо", "отлично", "подходит", 
      "good", "perfect", "looks great", "approve"), set next_step to "finalize"
    - If user requests corrections, improvements, reorganization, or has questions, 
      set next_step to "clarify"
    - Provide refined structure in the sections field
  </approval_detection>

  <refinement_principles>
    - Respond constructively to structural feedback (add/remove/reorder sections)
    - Adjust hierarchy depth based on user preferences
    - Refine section and subsection titles for better clarity
    - Ensure theses alignment with user's learning objectives
    - Maintain consistency with content parameters and target audience
  </refinement_principles>
</refinement_requirements>

<output_format>
  <language> {{ language }} </language>
</output_format>

<output_instruction>
Set next_step field based on user feedback:
- "finalize" if user is satisfied with structure
- "clarify" if further refinement is needed
</output_instruction>
"""

print("✓ HITL further prompt defined")

### 8.2. System Prompts для HITL

**Initial prompt** - используется как есть из cell-11 (`PLANNING_STRUCTURE_SYSTEM_PROMPT`)

**Further prompt** - с точечными изменениями для HITL-цикла: